# Time Series Exam Toolkit
Run the mock data generator cell first to create `mock_train.csv` and `mock_test.csv`. Then run the pipeline.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pmdarima as pm
from prophet import Prophet

import warnings
warnings.filterwarnings('ignore')

### 1. Mock Data Generator

In [24]:
def generate_mock_data():
    np.random.seed(42)
    dates = pd.date_range(start='2023-01-01', periods=120, freq='D')
    # Base trend and seasonality
    trend = np.linspace(10, 50, 120)
    seasonality = 10 * np.sin(np.arange(120) * (2 * np.pi / 7)) # Weekly seasonality
    
    # Exogenous variable (e.g., ad spend) that impacts the target
    exog = np.random.normal(50, 15, 120)
    exog_effect = exog * 0.3
    
    noise = np.random.normal(0, 5, 120)
    target = trend + seasonality + exog_effect + noise
    
    df = pd.DataFrame({'date': dates, 'target': target, 'promo_spend': exog})
    
    train = df.iloc[:-14]
    test = df.iloc[-14:]
    
    train.to_csv('mock_train.csv', index=False)
    # For the exam, they usually give you the test dates and exog variables, but hide the target
    test[['date', 'promo_spend']].to_csv('mock_test_exog.csv', index=False)
    test[['date', 'target']].to_csv('mock_test_actuals_hidden.csv', index=False) # For our evaluation
    print('Mock data generated: mock_train.csv, mock_test_exog.csv, mock_test_actuals_hidden.csv')

generate_mock_data()

Mock data generated: mock_train.csv, mock_test_exog.csv, mock_test_actuals_hidden.csv


### 2. Preprocessing & Evaluation Metrics

In [25]:
def load_and_prep(filepath, date_col='date', freq='D'):
    df = pd.read_csv(filepath)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col)
    df = df.asfreq(freq) # Critical for statsmodels
    df = df.ffill() # Quick imputation if needed
    return df

def evaluate_forecast(y_true, y_pred, metric='RMSE'):
    metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred)
    }
    print(f"--- Evaluation ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
    return metrics.get(metric.upper(), metrics['RMSE'])

### 3. Diagnostics (Stationarity & Seasonality)

In [26]:
def check_stationarity(timeseries):
    print('Results of Dickey-Fuller Test:')
    dftest = adfuller(timeseries.dropna(), autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    print(dfoutput)
    if dftest[1] <= 0.05:
        print("\nConclusion: Data is likely stationary (Reject null hypothesis).")
    else:
        print("\nConclusion: Data is non-stationary (Fail to reject null hypothesis). Differencing needed.")

def plot_diagnostics(timeseries, seasonal_period=7):
    fig, ax = plt.subplots(3, 1, figsize=(10, 12))
    timeseries.plot(ax=ax[0], title='Original Time Series')
    plot_acf(timeseries.dropna(), ax=ax[1], lags=40)
    plot_pacf(timeseries.dropna(), ax=ax[2], lags=40)
    plt.tight_layout()
    plt.show()
    
    # Decomposition
    decomp = seasonal_decompose(timeseries.dropna(), period=seasonal_period, model='additive')
    decomp.plot()
    plt.show()

### 4. Models (Holt-Winters, Auto-ARIMA/SARIMAX, Prophet)

In [27]:
def run_holt_winters(train_series, steps, seasonal_periods=7, trend='add', seasonal='add'):
    model = ExponentialSmoothing(train_series, trend=trend, seasonal=seasonal, seasonal_periods=seasonal_periods)
    fitted_model = model.fit()
    forecast = fitted_model.forecast(steps)
    return forecast

def run_auto_arima(train_series, steps, exog_train=None, exog_test=None, m=7):
    # m is the seasonal period (e.g., 7 for daily, 12 for monthly)
    model = pm.auto_arima(
        train_series, 
        X=exog_train,
        seasonal=True, m=m,
        stepwise=True, suppress_warnings=True, error_action="ignore"
    )
    print(model.summary())
    forecast = model.predict(n_periods=steps, X=exog_test)
    return forecast

def run_prophet(train_df, steps, target_col='target', date_col='date', exog_cols=None, test_exog_df=None):
    # Prophet requires strict column names 'ds' and 'y'
    prophet_df = train_df.reset_index()[[date_col, target_col]].rename(columns={date_col: 'ds', target_col: 'y'})
    
    m = Prophet(daily_seasonality=False, yearly_seasonality=False)
    
    if exog_cols:
        for col in exog_cols:
            prophet_df[col] = train_df[col].values
            m.add_regressor(col)
            
    m.fit(prophet_df)
    
    future = m.make_future_dataframe(periods=steps, freq='D')
    if exog_cols and test_exog_df is not None:
        # Append test exogenous variables to future dataframe
        future_exog = pd.concat([train_df[exog_cols], test_exog_df.set_index(date_col)[exog_cols]])
        for col in exog_cols:
            future[col] = future_exog[col].values
            
    forecast = m.predict(future)
    return forecast.set_index('ds').iloc[-steps:]['yhat']

### 5. Export Function

In [28]:
def export_submission(forecast_series, filename='submission.csv', index_label='date', value_label='prediction'):
    # Ensures format matches standard grading rubrics
    df_out = forecast_series.to_frame(name=value_label)
    df_out.index.name = index_label
    df_out.to_csv(filename)
    print(f"Saved predictions to {filename}")

### 6. Execution / Exam Workflow Demo

In [29]:
# 1. Load Data
train_df = load_and_prep('mock_train.csv')
test_exog_df = pd.read_csv('mock_test_exog.csv')
test_actuals = pd.read_csv('mock_test_actuals_hidden.csv') # For demo evaluation only
steps_to_forecast = len(test_exog_df)

# 2. Separate target and exog
y_train = train_df['target']
X_train = train_df[['promo_spend']]
X_test = test_exog_df.set_index('date')[['promo_spend']]

# 3. Run Diagnostics (uncomment during exam)
# check_stationarity(y_train)
# plot_diagnostics(y_train)

# 4. Generate Forecasts
print("Running Auto-ARIMA with Exogenous Variables...")
arima_preds = run_auto_arima(y_train, steps_to_forecast, exog_train=X_train, exog_test=X_test)

# 5. Evaluate (Assuming we have actuals to test locally before submission)
rmse = evaluate_forecast(test_actuals['target'].values, arima_preds)

# 6. Export Final Submission
export_submission(arima_preds, 'mock_group_submission.csv')

Running Auto-ARIMA with Exogenous Variables...
                                     SARIMAX Results                                      
Dep. Variable:                                  y   No. Observations:                  106
Model:             SARIMAX(0, 1, 1)x(2, 0, [], 7)   Log Likelihood                -350.111
Date:                            Thu, 05 Mar 2026   AIC                            710.221
Time:                                    00:52:58   BIC                            723.491
Sample:                                01-01-2023   HQIC                           715.598
                                     - 04-16-2023                                         
Covariance Type:                              opg                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
promo_spend     0.3601      0.045      7.935      0.000       0.2